In [1]:
# %%
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
)
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

import mne
mne.set_log_level("WARNING")

In [2]:
# %%
PROJECT_ROOT  = Path("..").resolve()
DERIVED_ROOT  = PROJECT_ROOT / "data" / "derived"
MANIFEST_PATH = DERIVED_ROOT / "manifests" / "manifest_spontaneous_validated.csv"
FEAT_A_PATH   = DERIVED_ROOT / "features" / "features_A_bandpower_epochwise.csv"
MATRICES_PATH = DERIVED_ROOT / "features" / "features_B_wpli_matrices_epoch_sliding.npz"
FULL_RESULTS_PATH = PROJECT_ROOT / "results" / "models" / "results_rf_nested_groupkfold_Aepoch_BwpliEpochSliding.csv"
OUT_DIR       = PROJECT_ROOT / "results" / "models"
FIGURE_DIR    = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# T3/T4 are the old 10-20 labels; modern equivalent used in this dataset is T7/T8
BANANA_CHANNELS = ["O1", "O2", "T7", "T8"]
BANDS           = ["theta", "alpha", "beta"]

OUTER_SPLITS = 5
INNER_SPLITS = 4
N_ITER       = 50   # more budget: model is tiny, each fit is fast
SEED         = 0

N_CORES = os.cpu_count() or 8
print(f"Cores: {N_CORES}")
print(f"Banana channels: {BANANA_CHANNELS}  (T3=T7, T4=T8 in modern 10-20 notation)")

Cores: 60
Banana channels: ['O1', 'O2', 'T7', 'T8']  (T3=T7, T4=T8 in modern 10-20 notation)


In [3]:
# %%
# ============================================
# Section 1. Get channel order from a sample file
# ============================================

manifest = pd.read_csv(MANIFEST_PATH)
sample_fp = manifest["file_path"].iloc[0]
sample_epochs = mne.io.read_epochs_eeglab(sample_fp, verbose="ERROR")
ch_names = sample_epochs.ch_names
ch_info  = sample_epochs.info

missing = [c for c in BANANA_CHANNELS if c not in ch_names]
assert not missing, f"Channels not found: {missing}"

banana_idx = [ch_names.index(c) for c in BANANA_CHANNELS]
print(f"All channels ({len(ch_names)}): {ch_names[:5]} … {ch_names[-3:]}")
print(f"Banana indices in 62-ch array: {dict(zip(BANANA_CHANNELS, banana_idx))}")

n_banana = len(banana_idx)
n_edges_banana = n_banana * (n_banana - 1) // 2
triu_rows_b, triu_cols_b = np.triu_indices(n_banana, k=1)
edge_labels = [
    f"{band}_{BANANA_CHANNELS[r]}-{BANANA_CHANNELS[c]}"
    for band in BANDS
    for r, c in zip(triu_rows_b, triu_cols_b)
]
print(f"Edges per band: {n_edges_banana}  |  Total features: {len(BANDS) * n_edges_banana}")
print(f"Edge labels: {edge_labels}")

All channels (62): ['Iz', 'O2', 'Oz', 'O1', 'PO8'] … ['Fp2', 'Fpz', 'Fp1']
Banana indices in 62-ch array: {'O1': 3, 'O2': 1, 'T7': 37, 'T8': 29}
Edges per band: 6  |  Total features: 18
Edge labels: ['theta_O1-O2', 'theta_O1-T7', 'theta_O1-T8', 'theta_O2-T7', 'theta_O2-T8', 'theta_T7-T8', 'alpha_O1-O2', 'alpha_O1-T7', 'alpha_O1-T8', 'alpha_O2-T7', 'alpha_O2-T8', 'alpha_T7-T8', 'beta_O1-O2', 'beta_O1-T7', 'beta_O1-T8', 'beta_O2-T7', 'beta_O2-T8', 'beta_T7-T8']


In [4]:
# %%
# ============================================
# Section 2. Build epoch-level feature matrix from NPZ matrices
# Subset each 62×62 matrix to the 4 banana channels, vectorize upper triangle
# ============================================

A = pd.read_csv(FEAT_A_PATH)
A_ok = A[A.get("extract_ok", True) == True].copy()
key_cols = ["subject_id", "recording_number", "epoch_index_original"]

npz = np.load(MATRICES_PATH)

feature_rows = []

for _, row in A_ok.iterrows():
    sid    = str(int(row["subject_id"]))
    recnum = int(row["recording_number"])
    eidx   = int(row["epoch_index_original"])

    edge_vec = []
    for band in BANDS:
        key = f"{sid}__rec{recnum}__e{eidx:04d}__{band}"
        mat = npz[key]                              # (62, 62)
        sub = mat[np.ix_(banana_idx, banana_idx)]   # (4, 4)
        edge_vec.append(sub[triu_rows_b, triu_cols_b])  # (6,)

    feature_rows.append(
        {"subject_id": row["subject_id"],
         "recording_number": recnum,
         "epoch_index_original": eidx,
         "drug": row["drug"],
         **dict(zip(edge_labels, np.concatenate(edge_vec)))}
    )

feat_df = pd.DataFrame(feature_rows)
print("Feature matrix shape:", feat_df.shape)
display(feat_df.head(3))

Feature matrix shape: (276, 22)


,subject_id,recording_number,epoch_index_original,drug,theta_O1-O2,theta_O1-T7,theta_O1-T8,theta_O2-T7,theta_O2-T8,theta_T7-T8,...,alpha_O1-T8,alpha_O2-T7,alpha_O2-T8,alpha_T7-T8,beta_O1-O2,beta_O1-T7,beta_O1-T8,beta_O2-T7,beta_O2-T8,beta_T7-T8
0,210,3,0,awake,0.330484,0.140510,0.127740,0.252628,0.242106,0.226394,...,0.221983,0.293132,0.352306,0.136991,0.174158,0.179617,0.257445,0.174751,0.179280,0.073721
1,210,3,1,awake,0.252401,0.063823,0.225399,0.272074,0.394202,0.261640,...,0.147420,0.355958,0.338376,0.359219,0.251929,0.079727,0.180622,0.252879,0.163554,0.139163
2,210,3,2,awake,0.144951,0.196079,0.082952,0.337465,0.265311,0.143445,...,0.076338,0.366129,0.303830,0.151505,0.260920,0.171814,0.230295,0.268142,0.273547,0.181204


In [5]:
# %%
# ============================================
# Section 3. Prepare X, y, groups
# ============================================

X = feat_df[edge_labels].to_numpy()
y = (feat_df["drug"] == "ketamine").astype(int).to_numpy()
groups = feat_df["subject_id"].astype(str).to_numpy()

print(f"X: {X.shape}  |  ketamine: {y.sum()}  awake: {(1-y).sum()}  |  subjects: {len(np.unique(groups))}")

X: (276, 18)  |  ketamine: 135  awake: 141  |  subjects: 10


In [6]:
# %%
# ============================================
# Section 4. Nested subject-aware RF — no PCA (only 18 features)
# Parallelized across (outer fold) tasks
# ============================================

param_dist = {
    "n_estimators":      randint(200, 1001),
    "max_depth":         [None, 5, 10, 20],
    "min_samples_split": randint(2, 11),
    "min_samples_leaf":  randint(1, 6),
    "max_features":      ["sqrt", "log2", None, 0.3, 0.5, 0.8],
}

outer_cv = GroupKFold(n_splits=OUTER_SPLITS)
splits   = list(outer_cv.split(X, y, groups))


def fit_eval_fold(
    fold: int, tr: np.ndarray, te: np.ndarray
) -> tuple[dict[str, Any], list[dict[str, Any]], dict[str, Any]]:
    inner_cv = GroupKFold(n_splits=INNER_SPLITS)
    gtr = groups[tr]

    rf = RandomForestClassifier(
        random_state=SEED, class_weight="balanced_subsample", n_jobs=1
    )
    rs = RandomizedSearchCV(
        rf, param_distributions=param_dist,
        n_iter=N_ITER, cv=inner_cv, scoring="balanced_accuracy",
        n_jobs=1, refit=True, random_state=SEED + fold, verbose=0,
    )
    rs.fit(X[tr], y[tr], groups=gtr)
    best = rs.best_estimator_

    proba = best.predict_proba(X[te])[:, 1]
    yhat  = (proba >= 0.5).astype(int)
    gte   = groups[te]

    try:
        auc = float(roc_auc_score(y[te], proba))
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y[te], yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (np.nan,) * 4

    fold_row = {
        "model": "B_banana_wpli",
        "fold": fold,
        "n_test": int(len(te)),
        "n_test_subjects": int(len(np.unique(gte))),
        "accuracy": float(accuracy_score(y[te], yhat)),
        "balanced_accuracy": float(balanced_accuracy_score(y[te], yhat)),
        "roc_auc": auc,
        "tn": float(tn), "fp": float(fp), "fn": float(fn), "tp": float(tp),
    }
    pred_rows = [
        {"model": "B_banana_wpli", "fold": fold,
         "subject_id": gte[i], "y_true": int(y[te][i]),
         "y_proba": float(proba[i]), "y_pred": int(yhat[i])}
        for i in range(len(te))
    ]
    param_row = {
        "model": "B_banana_wpli", "fold": fold,
        "best_score_inner": float(rs.best_score_),
        **rs.best_params_,
    }
    return fold_row, pred_rows, param_row


results = Parallel(n_jobs=min(N_CORES, OUTER_SPLITS), prefer="processes")(
    delayed(fit_eval_fold)(fold, tr, te)
    for fold, (tr, te) in enumerate(splits, start=1)
)

folds_banana  = pd.DataFrame([r[0] for r in results]).sort_values("fold").reset_index(drop=True)
preds_banana  = pd.DataFrame([pr for r in results for pr in r[1]])
params_banana = pd.DataFrame([r[2] for r in results])

print("Done.")
display(folds_banana)

Done.


,model,fold,n_test,n_test_subjects,accuracy,balanced_accuracy,roc_auc,tn,fp,fn,tp
0,B_banana_wpli,1,56,2,0.607143,0.607143,0.658163,14.0,14.0,8.0,20.0
1,B_banana_wpli,2,55,2,0.509091,0.507937,0.550265,16.0,12.0,15.0,12.0
2,B_banana_wpli,3,54,2,0.518519,0.510345,0.520000,18.0,11.0,15.0,10.0
3,B_banana_wpli,4,56,2,0.625000,0.625000,0.677296,19.0,9.0,12.0,16.0
4,B_banana_wpli,5,55,2,0.472727,0.470238,0.521164,17.0,11.0,18.0,9.0


In [7]:
# %%
# ============================================
# Section 5. Save results
# ============================================

folds_banana.to_csv(OUT_DIR / "results_rf_banana_wpli.csv", index=False)
preds_banana.to_csv(OUT_DIR / "predictions_rf_banana_wpli.csv", index=False)
params_banana.to_csv(OUT_DIR / "best_params_rf_banana_wpli.csv", index=False)
feat_df.to_csv(DERIVED_ROOT / "features" / "features_banana_wpli_edges.csv", index=False)
print("Saved.")

Saved.


In [8]:
# %%
# ============================================
# Section 6. Comparison: banana vs full B vs full A
# ============================================

full = pd.read_csv(FULL_RESULTS_PATH)

metrics = ["accuracy", "balanced_accuracy", "roc_auc"]

rows = []
for model_label, df_m in [
    ("A_bandpower_full62ch",      full[full["model"] == "A_bandpower_epoch"]),
    ("B_wpli_full62ch",           full[full["model"] == "B_wpli_edges_epoch_sliding"]),
    ("B_wpli_banana4ch (T7,T8,O1,O2)", folds_banana),
]:
    row = {"model": model_label, "n_features": None}
    for m in metrics:
        row[f"{m}_mean"] = float(df_m[m].mean())
        row[f"{m}_std"]  = float(df_m[m].std())
    rows.append(row)

rows[0]["n_features"] = 26
rows[1]["n_features"] = 5673
rows[2]["n_features"] = len(edge_labels)

comparison = pd.DataFrame(rows)
print("\n=== Model comparison ===")
display(comparison.set_index("model").round(4))


=== Model comparison ===


,n_features,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std
model,,,,,,,
A_bandpower_full62ch,26,0.7038,0.1458,0.7057,0.1481,0.8214,0.1586
B_wpli_full62ch,5673,0.5290,0.0757,0.5294,0.0792,0.5355,0.0682
"B_wpli_banana4ch (T7,T8,O1,O2)",18,0.5465,0.0661,0.5441,0.0679,0.5854,0.0764


In [9]:
# %%
# ============================================
# Section 7. Plot 1 — per-fold balanced accuracy comparison
# ============================================

plot_data = [
    ("A full (26 feat)",          full[full["model"] == "A_bandpower_epoch"]["balanced_accuracy"].values,       "steelblue"),
    ("B full (5673 feat)",        full[full["model"] == "B_wpli_edges_epoch_sliding"]["balanced_accuracy"].values, "coral"),
    ("B banana (18 feat)",        folds_banana["balanced_accuracy"].values,                                       "seagreen"),
]

fig, ax = plt.subplots(figsize=(9, 5))

for i, (label, vals, color) in enumerate(plot_data):
    jitter = (np.random.RandomState(i).rand(len(vals)) - 0.5) * 0.12
    ax.scatter(np.full(len(vals), i) + jitter, vals, color=color, s=60, zorder=3, label=label)
    ax.hlines(vals.mean(), i - 0.2, i + 0.2, colors=color, linewidths=2.5)

ax.axhline(0.5, linestyle="--", color="grey", linewidth=1, label="chance")
ax.set_xticks(range(len(plot_data)))
ax.set_xticklabels([p[0] for p in plot_data], fontsize=11)
ax.set_ylabel("Balanced accuracy (outer fold)", fontsize=12)
ax.set_title("Per-fold balanced accuracy: full vs. banana subset", fontsize=13)
ax.set_ylim(0.35, 1.0)
ax.legend(fontsize=10)
fig.tight_layout()

out = FIGURE_DIR / "banana_comparison_bacc.png"
fig.savefig(out, dpi=150)
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/banana_comparison_bacc.png


In [10]:
# %%
# ============================================
# Section 8. Plot 2 — mean ± std bar chart across all three metrics
# ============================================

model_labels = comparison["model"].tolist()
colors_bar   = ["steelblue", "coral", "seagreen"]
x = np.arange(len(metrics))
width = 0.22

fig, ax = plt.subplots(figsize=(10, 5))

for i, (label, color) in enumerate(zip(model_labels, colors_bar)):
    means = [comparison.loc[comparison["model"] == label, f"{m}_mean"].values[0] for m in metrics]
    stds  = [comparison.loc[comparison["model"] == label, f"{m}_std"].values[0]  for m in metrics]
    offset = (i - 1) * width
    ax.bar(x + offset, means, width, label=label, color=color, alpha=0.85,
           yerr=stds, capsize=4, error_kw={"linewidth": 1.5})

ax.axhline(0.5, linestyle="--", color="grey", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(["Accuracy", "Balanced accuracy", "ROC-AUC"], fontsize=12)
ax.set_ylabel("Score (mean ± std, 5 outer folds)", fontsize=12)
ax.set_title("Full vs. banana subset — classification metrics", fontsize=13)
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
fig.tight_layout()

out = FIGURE_DIR / "banana_comparison_all_metrics.png"
fig.savefig(out, dpi=150)
print("Saved:", out)
plt.close(fig)

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/banana_comparison_all_metrics.png


In [11]:
# %%
# ============================================
# Section 9. RF feature importances for the banana model
# Refit on the full dataset for a summary view
# ============================================

# Use median best params across folds
def _mode_or_median(s):
    try:
        return s.mode().iloc[0]
    except Exception:
        return s.median()

best_n_est  = int(round(params_banana["n_estimators"].median()))
best_depth  = params_banana["max_depth"].apply(lambda x: np.nan if x is None else x).median()
best_depth  = None if np.isnan(best_depth) else int(best_depth)
best_mf     = _mode_or_median(params_banana["max_features"])
best_mss    = int(round(params_banana["min_samples_split"].median()))
best_msl    = int(round(params_banana["min_samples_leaf"].median()))

rf_full = RandomForestClassifier(
    n_estimators=best_n_est, max_depth=best_depth,
    max_features=best_mf, min_samples_split=best_mss, min_samples_leaf=best_msl,
    random_state=SEED, class_weight="balanced_subsample", n_jobs=N_CORES,
)
rf_full.fit(X, y)

imp_df = pd.DataFrame({
    "edge": edge_labels,
    "importance": rf_full.feature_importances_,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(imp_df["edge"][::-1], imp_df["importance"][::-1], color="seagreen", alpha=0.85)
ax.set_xlabel("RF feature importance", fontsize=12)
ax.set_title("Banana wPLI feature importances\n(RF fit on full dataset with median best params)", fontsize=12)
fig.tight_layout()

out = FIGURE_DIR / "banana_feature_importances.png"
fig.savefig(out, dpi=150)
print("Saved:", out)
plt.close(fig)

display(imp_df.reset_index(drop=True))

Saved: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures/banana_feature_importances.png


,edge,importance
0,alpha_O1-T7,0.095212
1,theta_O1-O2,0.079632
2,theta_O1-T8,0.066273
3,alpha_O2-T8,0.066014
4,theta_O2-T7,0.057419
5,alpha_O1-O2,0.055742
6,alpha_O1-T8,0.053870
7,theta_O2-T8,0.053761
8,alpha_T7-T8,0.052724
9,beta_O1-T8,0.051720


In [12]:
# %%
print("\n=== Final comparison ===")
display(comparison.set_index("model")[["n_features",
    "balanced_accuracy_mean", "balanced_accuracy_std",
    "roc_auc_mean", "roc_auc_std"]].round(4))

print("\nAll figures saved to:", FIGURE_DIR)
for f in sorted(FIGURE_DIR.glob("banana_*.png")):
    print(" ", f.name)


=== Final comparison ===


,n_features,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std
model,,,,,
A_bandpower_full62ch,26,0.7057,0.1481,0.8214,0.1586
B_wpli_full62ch,5673,0.5294,0.0792,0.5355,0.0682
"B_wpli_banana4ch (T7,T8,O1,O2)",18,0.5441,0.0679,0.5854,0.0764



All figures saved to: /data/storage-occ-v2/repos/ketamine-prediction/ketamine-eeg-prediction/.claude/worktrees/cool-grothendieck-d69cd2/results/figures
  banana_comparison_all_metrics.png
  banana_comparison_bacc.png
  banana_feature_importances.png
